<a href="https://colab.research.google.com/github/mmbc560/GUIA2/blob/main/Laboratorio_previo_Oct_26.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:

# ============================================================
# UNIVERSIDAD EAN - TREND TECH
# LABORATORIO PREVIO AL DIAGRAMA DE ISHIKAWA
#
# DEL DIAGNÓSTICO INDIVIDUAL AL PROBLEMA Y SUS CAUSAS
#
# Docente: María Mercedes Bernal Cerquera, PhD.
#
# OBJETIVO:
# Ayudar a los equipos a unificar sus diagnósticos anteriores,
# elegir un problema y descubrir posibles causas mediante
# preguntas y un árbol de decisiones.
#
# MODALIDADES:
# 1. Ejemplo ficticio de AgroFresco
# 2. Proyecto real del equipo
#
# RESULTADOS:
# - Priorización de problemas
# - Árbol de selección del problema
# - Exploración guiada de posibles causas
# - Semáforos de causas
# - Árbol de investigación de causas
# - Mapa de oportunidades
# - Excel para utilizar posteriormente en Ishikawa
#
# IMPORTANTE:
# Las respuestas y calificaciones son hipótesis del equipo.
# Una puntuación alta no confirma una causa raíz.
# ============================================================

# En Google Colab, ejecutar si faltan librerías:
# !pip install pandas matplotlib networkx openpyxl

import os
import textwrap
import pandas as pd
import matplotlib.pyplot as plt
import networkx as nx

from IPython.display import display

# ------------------------------------------------------------
# CONFIGURACIÓN
# ------------------------------------------------------------

ROJO = "#E53935"
AMARILLO = "#F9A825"
VERDE = "#43A047"
AZUL = "#1565C0"
MORADO = "#7B1FA2"
GRIS = "#607D8B"

plt.rcParams["font.family"] = "DejaVu Sans"
plt.rcParams["figure.dpi"] = 125

CARPETA = "Trend_Tech_Pre_Ishikawa"
os.makedirs(CARPETA, exist_ok=True)


def titulo(mensaje):
    print("\n" + "=" * 65)
    print(mensaje)
    print("=" * 65)


def texto(mensaje):
    while True:
        r = input(mensaje).strip()
        if r:
            return r
        print("Escribe una respuesta.")


def numero(mensaje, minimo=1, maximo=5):
    while True:
        try:
            n = int(input(mensaje))
            if minimo <= n <= maximo:
                return n
        except ValueError:
            pass
        print(f"Escribe un número entre {minimo} y {maximo}.")


def si_no(mensaje):
    while True:
        r = input(mensaje + " (S/N): ").strip().upper()
        if r in ["S", "N"]:
            return r == "S"
        print("Responde S o N.")


def semaforo(valor):
    if valor >= 4:
        return "ROJO", ROJO
    if valor >= 3:
        return "AMARILLO", AMARILLO
    return "VERDE", VERDE


def guardar(nombre):
    plt.savefig(
        os.path.join(CARPETA, nombre),
        dpi=180,
        bbox_inches="tight",
        facecolor="white"
    )
    plt.show()
    plt.close()


def corto(cadena, ancho=25):
    return "\n".join(textwrap.wrap(str(cadena), ancho))


# ============================================================
# ETAPA 1. INTEGRACIÓN DE RESULTADOS INDIVIDUALES
# ============================================================

titulo("ETAPA 1 - REUNIÓN DEL EQUIPO")

print("""
Cada integrante debe compartir los resultados de la
actividad individual realizada en clases anteriores.

Revisen:
- Diagnóstico de Clientes Conectados
- Problemas identificados
- Síntomas
- Evidencias
- Posibles causas
- Alternativas de solución

No comiencen por elegir una aplicación.
Primero acuerden qué problema investigarán.
""")

print("\nMODALIDADES:")
print("1. Ejemplo de AgroFresco")
print("2. Proyecto de nuestro equipo")

modo = numero("Selecciona 1 o 2: ", 1, 2)

if modo == 1:

    empresa = "AgroFresco"

    problemas = [
        {
            "Problema": "Clientes no conocen productos disponibles",
            "Impacto": 5,
            "Frecuencia": 4,
            "Evidencia": 3,
            "Investigacion": 5
        },
        {
            "Problema": "Demoras en confirmar pedidos",
            "Impacto": 5,
            "Frecuencia": 5,
            "Evidencia": 4,
            "Investigacion": 5
        },
        {
            "Problema": "Dificultad para programar entregas",
            "Impacto": 4,
            "Frecuencia": 3,
            "Evidencia": 3,
            "Investigacion": 4
        },
        {
            "Problema": "Poco seguimiento después de la compra",
            "Impacto": 3,
            "Frecuencia": 3,
            "Evidencia": 2,
            "Investigacion": 4
        }
    ]

    print("\nCASO FICTICIO: AGROFRESCO")
    print("Empresa agrícola que vende hortalizas a clientes.")

else:

    empresa = texto("Nombre de la empresa: ")

    cantidad = numero(
        "¿Cuántos problemas candidatos compararán? (2-6): ",
        2, 6
    )

    problemas = []

    for i in range(cantidad):

        titulo(f"PROBLEMA CANDIDATO {i+1}")

        p = texto("Describe el problema observado: ")

        print("""
Califiquen de 1 a 5, mediante consenso:

Impacto: ¿Cuánto afecta al cliente?
Frecuencia: ¿Qué tan frecuente parece ser?
Evidencia: ¿Qué tan bien sustentado está?
Investigación: ¿Qué tan fácil es comprobarlo?

Si no conocen la frecuencia, indiquen una
valoración provisional y anoten que deben validarla.
""")

        problemas.append({
            "Problema": p,
            "Impacto": numero("Impacto (1-5): "),
            "Frecuencia": numero("Frecuencia (1-5): "),
            "Evidencia": numero("Evidencia (1-5): "),
            "Investigacion": numero("Facilidad de investigar (1-5): ")
        })


# ============================================================
# ETAPA 2. PRIORIZACIÓN DE PROBLEMAS
# ============================================================

df_problemas = pd.DataFrame(problemas)

criterios = [
    "Impacto", "Frecuencia",
    "Evidencia", "Investigacion"
]

# Ponderaciones iguales, como en el taller individual.
df_problemas["Puntaje"] = df_problemas[criterios].sum(axis=1)

df_problemas = df_problemas.sort_values(
    "Puntaje",
    ascending=False,
    kind="stable"
).reset_index(drop=True)

titulo("RESULTADOS: PRIORIZACIÓN DE PROBLEMAS")

display(df_problemas)

# Gráfica
orden = df_problemas.iloc[::-1]

plt.figure(figsize=(12, 6))

plt.barh(
    [corto(p, 36) for p in orden["Problema"]],
    orden["Puntaje"],
    color=AZUL
)

plt.xlim(0, 22)
plt.xlabel("Puntaje de priorización (máximo 20)")
plt.title(
    "TREND TECH | ¿QUÉ PROBLEMA INVESTIGAR PRIMERO?",
    fontweight="bold"
)

plt.grid(axis="x", alpha=0.2)
plt.tight_layout()
guardar("01_Priorizacion_Problemas.png")

print("""
INTERPRETACIÓN:

El problema mejor puntuado es un candidato para
investigar primero. No necesariamente es el único
problema importante.

El equipo debe comprobar que realmente afecta
a los clientes y que puede investigarlo.
""")


# ============================================================
# ETAPA 3. ÁRBOL DE DECISIÓN DEL PROBLEMA
# ============================================================

titulo("ETAPA 3 - ÁRBOL DE DECISIÓN")

print("""
Ahora revisaremos los problemas candidatos.

El árbol utilizará tres preguntas:

1. ¿Afecta al cliente de manera importante?
2. ¿Tenemos alguna evidencia?
3. ¿Podemos investigarlo?

Si alguna respuesta es NO, el programa indicará
qué falta antes de seleccionar ese problema.
""")

# El equipo responde para cada candidato.
# En el ejemplo se utilizan respuestas ficticias.

respuestas_ejemplo = [
    [True, True, True],
    [True, True, True],
    [True, True, True],
    [True, False, True]
]

decisiones = []

for i, fila in df_problemas.iterrows():

    print("\nPROBLEMA:", fila["Problema"])

    if modo == 1:
        # Respuestas ilustrativas para ejecutar el ejemplo.
        resp = respuestas_ejemplo[i]
    else:
        resp = [
            si_no("¿Afecta significativamente al cliente?"),
            si_no("¿Existe evidencia o información inicial?"),
            si_no("¿Es posible investigarlo?")
        ]

    pendientes = []

    if not resp[0]:
        pendientes.append("Revisar relevancia para el cliente")
    if not resp[1]:
        pendientes.append("Recopilar evidencia")
    if not resp[2]:
        pendientes.append("Definir cómo investigarlo")

    estado = (
        "CANDIDATO LISTO"
        if not pendientes
        else "REQUIERE REVISIÓN"
    )

    decisiones.append({
        "Problema": fila["Problema"],
        "Puntaje": fila["Puntaje"],
        "Afecta_cliente": resp[0],
        "Tiene_evidencia": resp[1],
        "Investigable": resp[2],
        "Estado": estado,
        "Pendientes": "; ".join(pendientes)
    })

df_decisiones = pd.DataFrame(decisiones)

display(df_decisiones)

# Seleccionar solamente candidatos que cumplen condiciones.
elegibles = df_decisiones[
    df_decisiones["Estado"] == "CANDIDATO LISTO"
].sort_values("Puntaje", ascending=False, kind="stable")

if elegibles.empty:
    print("""
Ningún problema cumple todas las condiciones.
Se necesita investigar más antes de elegir uno.
""")

    # Se permite explorar el mejor puntuado, pero se advierte
    # que no está validado.
    problema = df_problemas.iloc[0]["Problema"]
    estado_problema = "PROVISIONAL"

else:
    problema = elegibles.iloc[0]["Problema"]
    estado_problema = "CANDIDATO PRIORIZADO"

if modo == 2:
    print("\nPROBLEMA RECOMENDADO:", problema)
    if si_no("¿El equipo acuerda investigar este problema?"):
        pass
    else:
        print("\nSeleccionen el problema que investigarán:")
        for i, p in enumerate(df_problemas["Problema"], 1):
            print(f"{i}. {p}")
        idx = numero(
            "Número de problema: ",
            1, len(df_problemas)
        ) - 1
        problema = df_problemas.iloc[idx]["Problema"]
        estado_problema = "SELECCIÓN DEL EQUIPO"

titulo("PROBLEMA PARA EXPLORAR")
print(problema)
print("Estado:", estado_problema)


# ============================================================
# GRÁFICA: ÁRBOL DE SELECCIÓN
# ============================================================

G = nx.DiGraph()

nodos = [
    ("inicio", "Problemas\nidentificados"),
    ("impacto", "¿Afecta\nal cliente?"),
    ("evidencia", "¿Hay\nevidencia?"),
    ("investigar", "¿Se puede\ninvestigar?"),
    ("final", "Problema\npriorizado"),
    ("revisar", "Revisar\nrelevancia"),
    ("validar", "Buscar\nevidencia"),
    ("plan", "Preparar\ninvestigación")
]

G.add_nodes_from([x[0] for x in nodos])

aristas = [
    ("inicio", "impacto"),
    ("impacto", "evidencia"),
    ("impacto", "revisar"),
    ("evidencia", "investigar"),
    ("evidencia", "validar"),
    ("investigar", "final"),
    ("investigar", "plan")
]

G.add_edges_from(aristas)

pos = {
    "inicio": (0, 4),
    "impacto": (0, 3),
    "evidencia": (0, 2),
    "investigar": (0, 1),
    "final": (0, 0),
    "revisar": (2, 3),
    "validar": (2, 2),
    "plan": (2, 1)
}

colores_nodos = [
    VERDE if n == "final"
    else AMARILLO if n in ["revisar", "validar", "plan"]
    else AZUL
    for n in G.nodes()
]

plt.figure(figsize=(11, 8))

nx.draw_networkx(
    G, pos,
    labels=dict(nodos),
    node_color=colores_nodos,
    font_color="white",
    node_size=3400,
    font_size=9,
    arrows=True,
    arrowsize=20,
    edge_color=GRIS
)

nx.draw_networkx_edge_labels(
    G, pos,
    edge_labels={
        ("impacto", "evidencia"): "Sí",
        ("impacto", "revisar"): "No",
        ("evidencia", "investigar"): "Sí",
        ("evidencia", "validar"): "No",
        ("investigar", "final"): "Sí",
        ("investigar", "plan"): "No"
    },
    font_size=9
)

plt.title(
    "TREND TECH | ÁRBOL DE DECISIÓN DEL PROBLEMA",
    fontweight="bold"
)
plt.axis("off")
plt.tight_layout()
guardar("02_Arbol_Problema.png")


# ============================================================
# ETAPA 4. DESCUBRIR POSIBLES CAUSAS MEDIANTE PREGUNTAS
# ============================================================

titulo("ETAPA 4 - ¿POR QUÉ OCURRE EL PROBLEMA?")

print("""
Ahora no vamos a construir el Ishikawa todavía.

Vamos a investigar el problema mediante preguntas
sencillas sobre lo que experimentan los clientes.

Para cada situación, el equipo decidirá:

1. ¿Ocurre o podría ocurrir?
2. ¿Por qué podría estar ocurriendo?
3. ¿Qué impacto tendría?
4. ¿Qué evidencia existe?
5. ¿Qué oportunidad de mejora se abre?

Las respuestas serán posibles causas para investigar.
""")

# Preguntas organizadas según los cuatro subnodos
# de Clientes Conectados trabajados anteriormente.

bloques = {
    "Experiencia personalizada": [
        "¿Los clientes reciben atención poco adaptada a sus necesidades?",
        "¿La empresa desconoce las preferencias de sus clientes?"
    ],
    "Gestión de clientes": [
        "¿Se pierden o quedan sin seguimiento las solicitudes?",
        "¿Los clientes esperan demasiado para recibir respuesta?"
    ],
    "Pedidos sin fricciones": [
        "¿Es difícil solicitar, reservar o comprar un servicio?",
        "¿Los clientes desconocen el estado de sus pedidos?"
    ],
    "Servicios de campo conectados": [
        "¿Existen dificultades para coordinar citas o entregas?",
        "¿Los clientes reciben poca información sobre cambios?"
    ]
}

# Respuestas ficticias para AgroFresco.
# True = situación posible; False = no priorizada.
ejemplo_respuestas = [
    True, False,
    True, True,
    True, True,
    True, False
]

ejemplo_causas = [
    "Información de productos no personalizada",
    "",
    "Solicitudes dispersas en diferentes canales",
    "No existe responsable de confirmar pedidos",
    "Registro manual y poco claro de pedidos",
    "No se actualiza el estado de los pedidos",
    "Programación informal de las entregas",
    ""
]

ejemplo_impactos = [3, 0, 4, 5, 5, 4, 3, 0]
ejemplo_evidencias = [2, 0, 3, 3, 4, 3, 2, 0]

causas = []
contador = 0

for subnodo, preguntas in bloques.items():

    titulo("SUBNODO: " + subnodo.upper())

    for pregunta in preguntas:

        print("\nPREGUNTA:", pregunta)

        if modo == 1:
            ocurre = ejemplo_respuestas[contador]
        else:
            ocurre = si_no(
                "¿Esta situación ocurre o podría ocurrir?"
            )

        if ocurre:

            if modo == 1:
                causa = ejemplo_causas[contador]
                impacto = ejemplo_impactos[contador]
                evidencia = ejemplo_evidencias[contador]
            else:
                causa = texto(
                    "¿Por qué creen que ocurre? "
                    "Escriban una posible causa: "
                )

                impacto = numero(
                    "¿Qué impacto tendría? (1-5): "
                )

                evidencia = numero(
                    "¿Qué tan sólida es la evidencia? (1-5): "
                )

            # Pregunta de profundización.
            # Evita confundir síntomas con causas.
            print("\nPOSIBLE CAUSA:", causa)

            if modo == 1:
                causa_profunda = (
                    "Debe investigarse por qué existe: " + causa
                )
            else:
                causa_profunda = texto(
                    "¿Qué podría explicar esta causa? "
                    "(un nivel más profundo): "
                )

            # Una oportunidad no es todavía una solución.
            if modo == 1:
                oportunidad = (
                    "Mejorar el proceso relacionado con " + causa
                )
            else:
                oportunidad = texto(
                    "¿Qué aspecto se podría mejorar?: "
                )

            causas.append({
                "Subnodo": subnodo,
                "Pregunta": pregunta,
                "Causa": causa,
                "Causa_profunda": causa_profunda,
                "Impacto": impacto,
                "Evidencia": evidencia,
                "Oportunidad": oportunidad
            })

        contador += 1


# ============================================================
# ETAPA 5. PRIORIZAR CAUSAS SIN CONFUNDIR IMPACTO Y EVIDENCIA
# ============================================================

titulo("ETAPA 5 - CAUSAS PRIORITARIAS")

if not causas:
    print("""
No se identificaron causas mediante estas preguntas.

El equipo debe ampliar la investigación o formular
otras preguntas relacionadas con su problema.
""")

    df_causas = pd.DataFrame(columns=[
        "Subnodo", "Pregunta", "Causa", "Causa_profunda",
        "Impacto", "Evidencia", "Oportunidad",
        "Semaforo", "Estado"
    ])

else:

    df_causas = pd.DataFrame(causas)

    df_causas["Semaforo"] = df_causas["Impacto"].apply(
        lambda x: semaforo(x)[0]
    )

    # Impacto y evidencia se mantienen separados.
    # Una causa puede tener alto impacto y poca evidencia.
    def estado_causa(fila):
        if fila["Impacto"] >= 4 and fila["Evidencia"] >= 3:
            return "PRIORITARIA PARA VALIDAR"
        if fila["Impacto"] >= 4 and fila["Evidencia"] < 3:
            return "ALTO IMPACTO, FALTA EVIDENCIA"
        return "SEGUIMIENTO"

    df_causas["Estado"] = df_causas.apply(
        estado_causa,
        axis=1
    )

    df_causas = df_causas.sort_values(
        ["Impacto", "Evidencia"],
        ascending=False,
        kind="stable"
    ).reset_index(drop=True)

    display(df_causas)

    # Gráfica de semáforos
    orden = df_causas.iloc[::-1]

    colores = [
        semaforo(x)[1]
        for x in orden["Impacto"]
    ]

    plt.figure(figsize=(12, 7))

    plt.barh(
        [corto(c, 38) for c in orden["Causa"]],
        orden["Impacto"],
        color=colores
    )

    plt.xlim(0, 5.5)
    plt.xlabel("Impacto estimado (1-5)")
    plt.title(
        "TREND TECH | SEMÁFOROS DE POSIBLES CAUSAS",
        fontweight="bold"
    )

    plt.grid(axis="x", alpha=0.2)
    plt.tight_layout()
    guardar("03_Semaforos_Causas.png")

    print("""
INTERPRETACIÓN:

ROJO: causa de impacto alto estimado.
AMARILLO: impacto medio.
VERDE: impacto bajo.

Las causas con impacto alto y evidencia suficiente
son candidatas prioritarias para investigar.

Las causas con impacto alto pero poca evidencia
necesitan verificación antes de asumirlas como reales.
""")


# ============================================================
# ETAPA 6. ÁRBOL DE POSIBLES CAUSAS
# ============================================================

if not df_causas.empty:

    titulo("ETAPA 6 - ÁRBOL DE INVESTIGACIÓN DE CAUSAS")

    # Se muestran hasta cinco causas para evitar
    # una gráfica excesivamente saturada.
    principales = df_causas.head(5)

    G2 = nx.DiGraph()

    G2.add_node("P", label=corto(problema, 27))

    pos2 = {"P": (0, 0)}
    colores2 = [AZUL]

    total = len(principales)

    for i, fila in principales.iterrows():

        x = i - (total - 1) / 2

        id_causa = f"C{i}"
        id_profunda = f"D{i}"

        G2.add_node(
            id_causa,
            label=corto(fila["Causa"], 22)
        )

        G2.add_node(
            id_profunda,
            label=corto(fila["Causa_profunda"], 22)
        )

        G2.add_edge("P", id_causa)
        G2.add_edge(id_causa, id_profunda)

        pos2[id_causa] = (x * 3.2, -2)
        pos2[id_profunda] = (x * 3.2, -4)

    etiquetas = nx.get_node_attributes(G2, "label")

    colores2 = []

    for nodo in G2.nodes():

        if nodo == "P":
            colores2.append(AZUL)
        elif nodo.startswith("C"):
            indice = int(nodo[1:])
            impacto = principales.iloc[indice]["Impacto"]
            colores2.append(semaforo(impacto)[1])
        else:
            colores2.append(MORADO)

    plt.figure(figsize=(17, 9))

    nx.draw_networkx(
        G2,
        pos2,
        labels=etiquetas,
        node_color=colores2,
        font_color="white",
        font_size=8,
        node_size=4800,
        edge_color=GRIS,
        arrows=True,
        arrowsize=17
    )

    plt.title(
        "TREND TECH | ÁRBOL DE POSIBLES CAUSAS\n"
        "Del problema a explicaciones que deben investigarse",
        fontweight="bold"
    )

    plt.axis("off")
    plt.tight_layout()
    guardar("04_Arbol_Causas.png")


# ============================================================
# ETAPA 7. IDENTIFICAR OPORTUNIDADES
# ============================================================

titulo("ETAPA 7 - ¿QUÉ OPORTUNIDADES SE ABREN?")

if not df_causas.empty:

    print("""
Una causa puede abrir diferentes oportunidades.

Por ejemplo:

CAUSA:
Los pedidos no se confirman oportunamente.

OPORTUNIDADES:
- Mejorar la comunicación.
- Organizar las solicitudes.
- Facilitar la confirmación.
- Informar el estado del pedido.

Todavía no necesitamos elegir la tecnología.
""")

    oportunidades = df_causas[
        ["Causa", "Oportunidad", "Impacto", "Estado"]
    ].copy()

    display(oportunidades)

    # Gráfica de oportunidades vinculadas a causas.
    top = df_causas.head(5)

    fig, ax = plt.subplots(figsize=(15, 7))
    ax.set_xlim(0, 10)
    ax.set_ylim(-0.5, len(top) + 0.5)
    ax.axis("off")

    ax.text(
        2.2, len(top) + 0.2,
        "POSIBLE CAUSA",
        ha="center",
        fontweight="bold",
        fontsize=13,
        color=AZUL
    )

    ax.text(
        7.8, len(top) + 0.2,
        "OPORTUNIDAD DE MEJORA",
        ha="center",
        fontweight="bold",
        fontsize=13,
        color=VERDE
    )

    for i, fila in top.iterrows():

        y = len(top) - i - 0.5

        ax.text(
            2.2, y,
            corto(fila["Causa"], 32),
            ha="center",
            va="center",
            fontsize=9,
            bbox=dict(
                boxstyle="round,pad=0.6",
                facecolor="#E3F2FD",
                edgecolor=AZUL
            )
        )

        ax.annotate(
            "",
            xy=(6.0, y),
            xytext=(4.2, y),
            arrowprops=dict(
                arrowstyle="->",
                color=GRIS,
                lw=2
            )
        )

        ax.text(
            7.8, y,
            corto(fila["Oportunidad"], 34),
            ha="center",
            va="center",
            fontsize=9,
            bbox=dict(
                boxstyle="round,pad=0.6",
                facecolor="#E8F5E9",
                edgecolor=VERDE
            )
        )

    plt.title(
        "UNIVERSIDAD EAN | TREND TECH\n"
        "DEL ANÁLISIS DE CAUSAS A LAS OPORTUNIDADES",
        fontweight="bold",
        pad=25
    )

    plt.tight_layout()
    guardar("05_Mapa_Oportunidades.png")


# ============================================================
# ETAPA 8. PREPARAR INSUMOS PARA ISHIKAWA
# ============================================================

titulo("ETAPA 8 - PREPARACIÓN DEL ISHIKAWA")

print("\nEMPRESA:", empresa)
print("PROBLEMA SELECCIONADO:", problema)

if not df_causas.empty:

    print("\nPOSIBLES CAUSAS PRIORITARIAS:")

    for i, fila in df_causas.head(5).iterrows():

        print(f"\n{i+1}. {fila['Causa']}")
        print("Impacto:", fila["Impacto"])
        print("Evidencia:", fila["Evidencia"])
        print("Estado:", fila["Estado"])
        print("Posible explicación:", fila["Causa_profunda"])

print("""
SIGUIENTE ACTIVIDAD:

Construir el diagrama de Ishikawa utilizando
el problema seleccionado y las posibles causas.

Las causas deberán organizarse en las seis
categorías del taller anterior:

1. Personas
2. Métodos
3. Tecnología
4. Información
5. Recursos
6. Entorno

No es necesario utilizar todas las categorías
si no existen causas pertinentes.

Las causas deben validarse con la empresa.
""")


# ============================================================
# ETAPA 9. EXPORTAR RESULTADOS
# ============================================================

archivo = os.path.join(
    CARPETA,
    "Diagnostico_Pre_Ishikawa.xlsx"
)

with pd.ExcelWriter(archivo, engine="openpyxl") as writer:

    df_problemas.to_excel(
        writer,
        sheet_name="Problemas",
        index=False
    )

    df_decisiones.to_excel(
        writer,
        sheet_name="Arbol_Decision",
        index=False
    )

    df_causas.to_excel(
        writer,
        sheet_name="Posibles_Causas",
        index=False
    )

    if not df_causas.empty:
        oportunidades.to_excel(
            writer,
            sheet_name="Oportunidades",
            index=False
        )

titulo("LABORATORIO FINALIZADO")

print("""
RESULTADOS:

01_Priorizacion_Problemas.png
02_Arbol_Problema.png
03_Semaforos_Causas.png
04_Arbol_Causas.png
05_Mapa_Oportunidades.png
Diagnostico_Pre_Ishikawa.xlsx

Los archivos se guardan en la carpeta:
Trend_Tech_Pre_Ishikawa

NOTA:
Si no se identificaron causas, únicamente se
generan las gráficas y tablas correspondientes
a las etapas que pudieron completarse.
""")
